We will use the notebook to answer a few concrete questions:
1. What does the MCP response actually look like?
2. What information does data360_get_metadata return?
3. How are units, missing observations, and dimensions represented?
4. Can we transform the observations into our standardized schema without losing information?

In [19]:
from pathlib import Path
import sys

# Locate the directory containing agentic_framework
root = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "agentic_framework" / "llm_client.py").is_file()
    ),
    None,
)

if root is None:
    raise FileNotFoundError("Cannot locate agentic_framework")

sys.path.insert(0, str(root))

print("Project root:", root)

Project root: c:\Users\saiou\Desktop\mapping_visualisation_project\world-population-interactive-visualisation


In [20]:
from mcp import Client
from agentic_framework.mcp_client import MCP_URL

async with Client(MCP_URL) as client:
    tools = await client.list_tools()

    for tool in tools.tools:
        if tool.name in {
            "data360_search_indicators",
            "data360_get_metadata",
            "data360_get_data",
            "data360_get_disaggregation",
        }:
            print(f"\nTool: {tool.name}")
            print("Input schema:", tool.input_schema)


Tool: data360_search_indicators
Input schema: {'additionalProperties': False, 'properties': {'query': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Single topic query (e.g. "unemployment"). Use ONLY for a single topic. Do NOT combine multiple topics with \'and\' or \'or\' (e.g. do NOT use query="population and life expectancy"). Avoid special characters like parentheses () or dollar signs $. Example: \'GDP per capita\'.'}, 'required_country': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Semicolon-separated ISO country codes (e.g. "KEN;USA"). Shared across all queries in \'query\' or \'queries\'. Consider calling `data360_expand_country_group` to find country codes in regional/income groups, or `data360_find_codelist_value` to resolve country names.'}, 'limit': {'default': 5, 'type': 'integer', 'description': 'Max indicators per query (default 5).'}, 'offset': {'default': 0, 'type': 'integer', 'description': 'Of

assume that the query of the user was "what is the gdp in switzerland since 2015?

for that, qwen should search for gdp per capita, the goal is to discover what indicators exist in the worldbank that is related to gdp,
here, we are playing manually the role of qwen, we ask which indicators exist ourselves, we validate them, 
and we choose which one we will use, then we pass this information unto mcp to actually get the data we need, in the following cell, 
we execute the "which indicators are there", and in the next cell, we use the indicator we chose to perform the actual data retreival!

In [27]:
import json
from mcp import Client
from agentic_framework.mcp_client import MCP_URL

async with Client(MCP_URL) as client:

    search_result = await client.call_tool(
        "data360_search_indicators",
        {
            "query": "GDP per capita",
            "required_country": "CHE",
            "database": "wdi",
            "limit": 5
        }
    )
    print(type(search_result))
    print(search_result)

print("Tool error:", search_result.is_error)

if search_result.structured_content is not None:
    print("Structured content:")
    print(json.dumps(
        search_result.structured_content,
        indent=2,
        default=str
    )[:10000])
else:
    print("No structured content.")
    for item in search_result.content:
        if hasattr(item, "text"):
            print(item.text[:10000])

<class 'mcp_types._types.CallToolResult'>
meta=None content=[TextContent(type='text', text='{"count":5,"total_count":200,"offset":0,"has_more":true,"next_offset":5,"indicators":[{"idno":"WB_WDI_NY_GDP_PCAP_CD","database_id":"WB_WDI","database_name":"World Development Indicators (WDI)","name":"GDP per capita (current US$)","truncated_definition":"Gross domestic product is the total income earned through the production of goods and services in an","periodicity":"Annual","latest_data":"2024","time_period_range":"1960-2024","covers_country":{"CHE":true},"requested_country":"CHE","dimensions":null,"primary_source_of":null},{"idno":"WB_WDI_NY_GDP_PCAP_KD","database_id":"WB_WDI","database_name":"World Development Indicators (WDI)","name":"GDP per capita (constant 2015 US$)","truncated_definition":"Gross domestic product is the total income earned through the production of goods and services in an","periodicity":"Annual","latest_data":"2024","time_period_range":"1960-2024","covers_country":{"C

retreive metadata and observations, from the indicator we selected by asking what indicators are there to answer the question

In [22]:
import json
from mcp import Client
from agentic_framework.mcp_client import MCP_URL

indicator_id = "WB_WDI_NY_GDP_PCAP_CD"
database_id = "WB_WDI"

async with Client(MCP_URL) as client:

    metadata_result = await client.call_tool(
        "data360_get_metadata",
        {
            "database_id": database_id,
            "indicator_id": indicator_id,
            "required_country": "CHE",
            "fetch_disaggregation": True,
        },
    )

    data_result = await client.call_tool(
        "data360_get_data",
        {
            "database_id": database_id,
            "indicator_id": indicator_id,
            "country_code": "CHE",
            "start_year": 2015,
            "end_year": 2023,
            "limit": 100,
            "offset": 0,
        },
    )


def extract_response(result):
    if result.is_error:
        raise RuntimeError(str(result.content))

    if result.structured_content is not None:
        return result.structured_content

    texts = [
        item.text
        for item in result.content
        if item.type == "text"
    ]

    if len(texts) != 1:
        raise ValueError("Unexpected MCP response format.")

    return json.loads(texts[0])


metadata = extract_response(metadata_result)
observations_response = extract_response(data_result)

print("=== METADATA ===")
print(json.dumps(metadata, indent=2, default=str)[:12000])

print("\n=== OBSERVATIONS ===")
print(json.dumps(
    observations_response,
    indent=2,
    default=str
)[:12000])

=== METADATA ===
{
  "indicator_metadata": {
    "idno": "WB_WDI_NY_GDP_PCAP_CD",
    "doi": null,
    "name": "GDP per capita (current US$)",
    "display_name": null,
    "database_id": "WB_WDI",
    "database_name": "World Development Indicators (WDI)",
    "date_last_update": null,
    "date_released": null,
    "measurement_unit": "current US$",
    "release_calendar": null,
    "periodicity": "Annual",
    "base_period": null,
    "definition_short": null,
    "definition_long": "Gross domestic product is the total income earned through the production of goods and services in an economic territory during an accounting period. It can be measured in three different ways: using either the expenditure approach, the income approach, or the production approach. The core indicator has been divided by the general population to achieve a per capita estimate.This indicator is expressed in current prices, meaning no adjustment has been made to account for price changes over time. This indic

now, we can use these definitions to establish normalise metadata, then, do the normalisation in transform.py, 

In [23]:
from agentic_framework.transform import (
    normalize_metadata,
    normalize_observations,
)

indicators_df = normalize_metadata(metadata)

observations_df = normalize_observations(
    observations_response,
    "WB_WDI_NY_GDP_PCAP_CD",
)

display(indicators_df)
display(observations_df.sort_values("year"))

,indicator_id,database_id,name,unit,definition,periodicity
0,WB_WDI_NY_GDP_PCAP_CD,WB_WDI,GDP per capita (current US$),current US$,Gross domestic product is the total income ear...,Annual


,indicator_id,country_iso3,year,value
8,WB_WDI_NY_GDP_PCAP_CD,CHE,2015,85265.062052
7,WB_WDI_NY_GDP_PCAP_CD,CHE,2016,83432.411160
6,WB_WDI_NY_GDP_PCAP_CD,CHE,2017,83610.248611
5,WB_WDI_NY_GDP_PCAP_CD,CHE,2018,87002.338229
4,WB_WDI_NY_GDP_PCAP_CD,CHE,2019,85872.970230
3,WB_WDI_NY_GDP_PCAP_CD,CHE,2020,87529.701639
2,WB_WDI_NY_GDP_PCAP_CD,CHE,2021,96582.868708
1,WB_WDI_NY_GDP_PCAP_CD,CHE,2022,97809.095567
0,WB_WDI_NY_GDP_PCAP_CD,CHE,2023,104449.754518


In [24]:
from agentic_framework.analytics import (
    create_connection,
    get_schema,
)

con = create_connection({
    "observations": observations_df,
    "indicators": indicators_df,
})

print(get_schema(
    con,
    approved_tables={"observations", "indicators"},
))

CREATE TABLE "indicators" (
    "indicator_id" VARCHAR,
    "database_id" VARCHAR,
    "name" VARCHAR,
    "unit" VARCHAR,
    "definition" VARCHAR,
    "periodicity" VARCHAR
);

CREATE TABLE "observations" (
    "indicator_id" VARCHAR,
    "country_iso3" VARCHAR,
    "year" BIGINT,
    "value" DOUBLE
);


In [25]:
result = con.execute("""
    SELECT
        o.year,
        o.value,
        i.unit
    FROM observations AS o
    JOIN indicators AS i
        ON o.indicator_id = i.indicator_id
    ORDER BY o.year
""").df()

display(result)
con.close()

,year,value,unit
0,2015,85265.062052,current US$
1,2016,83432.411160,current US$
2,2017,83610.248611,current US$
3,2018,87002.338229,current US$
4,2019,85872.970230,current US$
5,2020,87529.701639,current US$
6,2021,96582.868708,current US$
7,2022,97809.095567,current US$
8,2023,104449.754518,current US$
